In [1]:
import hashlib
from pathlib import Path
import json, subprocess, re
from collections import defaultdict

import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from datetime import datetime, date

ROOT = Path.cwd().parent
DATA_DIR = ROOT / "data" / "raw"
DATA_CONFIG = ROOT / "data" / "monitoring_config" / "test_id_to_target_host_mapper.json"

ADBLOCK_NODE_SCRIPT = ROOT / "shared" / "adblocker_ghostery" / "adblock.mjs"
ADBLOCK_CACHE_BLOCKED_URLS_FILE = ROOT / "shared" / "adblock_cache" / "blocked_urls.txt"
ADBLOCK_CACHE_PROCESSED_FILES_FILE = ROOT / "shared" / "adblock_cache" / "processed_files.txt"

GRAPH_SAVE_FOLDER = './results'
GRAPH_RESULTS_DIR = Path(GRAPH_SAVE_FOLDER)

# Sliding window parameters
WINDOW = 288  # history window size in snapshots (288 × 10 min = 48 h)
OUTAGE_CONSEC = 3  # consecutive missing snapshots to declare MISSING_RESOURCE (3 × 10 min = 30 min)
OUTAGE_MIN_PRES = 0.99  # minimum presence ratio in window to be eligible for MISSING_RESOURCE
NEW_RES_FUTURE_CONSEC = 6  # consecutive present snapshots to declare NEW_RESOURCE (6 × 10 min = 1 h)
MAX_RESOURCES_TO_ANALYZE = 100  # maximum unique resources per website to analyze 

# Visualisation parameters
RESOURCE_URL_HEAD_SIZE = 200 # Maximum number of characters preserved from the beginning of a URL.
RESOURCE_URL_TAIL_SIZE = 12  # Number of characters preserved from the end of a truncated URL

# Anomaly marker colours for plots
ANOM_COLORS = {
    'MISSING_RESOURCE': '#ef4444',  # red
    'NEW_RESOURCE': '#22c55e',  # green
    'ERROR_4XX': '#f97316',  # orange
    'ERROR_5XX': '#a855f7',  # purple
}


# HELPER FUNCTIONS

def make_cache_key(url, source_url, type_):
    # composite key from URL, source and type, null byte as separator
    raw = f"{url}\x00{source_url}\x00{type_}"
    # md5 hash of key
    return hashlib.md5(raw.encode()).hexdigest()


# same logic as make_cache_key but without hashing
def get_composite_key_of_resource(r):
    return (
            r.get("url", "") + "\x00" +
            r.get("sourceUrl", "") + "\x00" +
            r.get("type", "other")
    )


# loads md5 hashes of blocked URLs from cache file into set
def load_adblock_cache():
    print(f"Loading blocked urls from cache file ... {ADBLOCK_CACHE_BLOCKED_URLS_FILE}")
    if not ADBLOCK_CACHE_BLOCKED_URLS_FILE.exists():
        ADBLOCK_CACHE_BLOCKED_URLS_FILE.parent.mkdir(parents=True, exist_ok=True)
        ADBLOCK_CACHE_BLOCKED_URLS_FILE.touch()
        print("  [CACHE] blocked_urls.txt not found, created empty file.")
        return set()
    return set(open(ADBLOCK_CACHE_BLOCKED_URLS_FILE, encoding='utf-8').read().splitlines())


# loads the list of already processed files. used to skip reprocessing
def load_adblock_processed_files_cache():
    if not ADBLOCK_CACHE_PROCESSED_FILES_FILE.exists():
        ADBLOCK_CACHE_PROCESSED_FILES_FILE.parent.mkdir(parents=True, exist_ok=True)
        ADBLOCK_CACHE_PROCESSED_FILES_FILE.touch()
        print("  [CACHE] processed_files.txt not found, created empty file.")
        return set()
    return set(open(ADBLOCK_CACHE_PROCESSED_FILES_FILE, encoding='utf-8').read().splitlines())


def adblock_store_to_cache(files):
    if not ADBLOCK_NODE_SCRIPT.exists():
        return

    for file in files:
        resources = set()
        with open(file, encoding='utf-8') as f:
            print(f"Adblocker loading to cache ... {file}")
            for raw in f:
                raw = raw.strip()
                if not raw:
                    continue
                try:
                    obj = json.loads(raw)
                except json.JSONDecodeError:
                    continue

                result = obj.get('Result')
                if not result or result.get('status') != 'completed':
                    continue

                pages = result.get('visited_pages', [])
                if not pages:
                    continue

                for res in pages[0].get('waterfall_analysis', []):
                    url = res.get('url', '')
                    if not url.startswith('http'):
                        continue
                    # collect unique composite keys for all HTTP resources in this file
                    resources.add(get_composite_key_of_resource(res))

            # pass all resource keys to the Node.js adblock via stdin    
            stdin_data = '\n'.join(resources) + '\n'
            try:
                r = subprocess.run(
                    ['node', str(ADBLOCK_NODE_SCRIPT)],
                    input=stdin_data,
                    capture_output=True,
                    text=True,
                    timeout=240
                )
            except (FileNotFoundError, subprocess.TimeoutExpired) as e:
                print(f'  [ERROR] adblock: {e}')
                return
            # node returns only the composite keys that were blocked one per line 
            blocked_urls = {
                line.strip()
                for line in r.stdout.splitlines()
                if line.strip()
            }
            # mark file as processed to ignore in repeating runs
            with open(ADBLOCK_CACHE_PROCESSED_FILES_FILE, 'a') as pf:
                pf.write(file.name + '\n')

            # store md5 hashes of blocked composite keys for later use
            with open(ADBLOCK_CACHE_BLOCKED_URLS_FILE, 'a') as bf:
                for key in blocked_urls:
                    bf.write(hashlib.md5(key.encode()).hexdigest() + '\n')


def status_class(code):
    try:
        c = int(code)
        if 400 <= c < 500:
            return '4xx'
        if 500 <= c < 600:
            return '5xx'
    except (ValueError, TypeError):
        pass
    return 'ok'


print('Configuration OK.')

Configuration OK.


In [2]:
# LOADING CONFIG AND DISCOVERY OF RAW DATASET 


with open(DATA_CONFIG, encoding='utf-8') as f:
    data_config = json.load(f)

# build  mappings test_id -> target_host
testid_to_host = {e['test_id']: e['target_host'] for e in data_config}
host_to_testid = {e['target_host']: e['test_id'] for e in data_config}

# discover all available JSON data files and their dates from data/raw directory
json_files = sorted(DATA_DIR.glob("webapp.http_dynamic.single_page.tranco.*.json"))


def extract_date(p: Path) -> str | None:
    # extract date string (YYYY-MM-DD) from a file name."""
    m = re.search(r'(\d{4}-\d{2}-\d{2})', p.stem)
    return m.group(1) if m else None


available_dates = sorted({d for p in json_files if (d := extract_date(p))})

print(f"Available date range: {available_dates[0]} → {available_dates[-1]}")
print(f"Sites loaded from config: {len(data_config)}")

Available date range: 2026-02-24 → 2026-02-28
Sites loaded from config: 62


In [3]:
from matplotlib.colors import LinearSegmentedColormap


# AGGREGATION

def aggregate_page(snapshots, blocked=None):
    """
    For each snapshot, aggregate per-resource presence and error counts.
    Optionally filters ads/trackers 

    Returns:
        snap_data : list of dicts  {resource_id: {present, 4xx, 5xx}}
        resources : list of seen resources
        n         : total number of snapshots
    """

    # remove blocked resources based on hash match of resource and cache file entries
    if blocked:
        for snap in snapshots:
            snap['resources'] = [
                r for r in snap['resources']
                if make_cache_key(r['url'], r.get('source', ''), r.get('type', 'other'))
                   not in blocked
            ]

    n = len(snapshots)
    all_resources = set()
    snap_data = []
    for snap in snapshots:
        per_resource = defaultdict(lambda: {'present': 0, '4xx': 0, '5xx': 0})
        for r in snap['resources']:
            resource_id = f"{r['method']}||{r['url']}"
            if not resource_id:
                continue
            all_resources.add(resource_id)
            per_resource[resource_id]['present'] = 1
            sc = status_class(r['status'])
            if sc in ('4xx', '5xx'):
                per_resource[resource_id][sc] += 1
        snap_data.append(dict(per_resource))

    # sort resources by how many snapshots they appear in
    resources = sorted(
        all_resources,
        key=lambda d: sum(1 for s in snap_data if d in s),
        reverse=True, )

    return snap_data, resources, n


print('Aggregation function loaded.')


# ─── SLIDING WINDOW ANOMALY DETECTION ─────────────────────────────────────────

def detect_sliding_window(snap_data, resources, n_snaps):
    """
    Detect anomalies using a sliding window.

    MISSING_RESOURCE : resource present in >= 99 % of the last WINDOW snapshots (2 days),
                     then missing for OUTAGE_CONSEC consecutive snapshots (30 min).
    NEW_RESOURCE     : resource absent for the entire WINDOW, then present for
                     NEW_RES_FUTURE_CONSEC consecutive snapshots (1 h).
    ERROR_4XX/5XX  : current count exceeds local median baseline,
                     or baseline is zero and any error occurs.

    Returns:
        anomalies : dict  {resource_id: [(snap_index, anomaly_type), …]}
        presence  : dict  {resource_id: np.array of 0/1 per snapshot}
        err_4xx   : dict  {resource_id: np.array of 4xx counts per snapshot}
        err_5xx   : dict  {resource_id: np.array of 5xx counts per snapshot}
    """

    presence = {}
    err_4xx = {}
    err_5xx = {}

    for res in resources:
        presence[res] = np.array(
            [1.0 if (res in s and s[res]['present']) else 0.0 for s in snap_data],
            dtype=np.float32
        )
        err_4xx[res] = np.array(
            [s[res]['4xx'] if res in s else 0 for s in snap_data],
            dtype=np.float32
        )
        err_5xx[res] = np.array(
            [s[res]['5xx'] if res in s else 0 for s in snap_data],
            dtype=np.float32
        )

    anomalies = defaultdict(list)

    for dom in resources:
        pres = presence[dom]
        e4 = err_4xx[dom]
        e5 = err_5xx[dom]

        for t in range(WINDOW, n_snaps):
            window_pres = pres[t - WINDOW: t]
            baseline_pres = float(np.mean(window_pres))

            # MISSING_RESOURCE 
            # condition: resource present in >= OUTAGE_MIN_PRES  of history window,
            #            then missing for OUTAGE_CONSEC consecutive snapshots
            if baseline_pres >= OUTAGE_MIN_PRES:
                end = min(t + OUTAGE_CONSEC, n_snaps)
                consec_missing = pres[t:end]
                if (len(consec_missing) == OUTAGE_CONSEC
                        and float(np.sum(consec_missing)) == 0
                        and (t == WINDOW or pres[t - 1] == 1.0)):
                    anomalies[dom].append((t, 'MISSING_RESOURCE'))

            # NEW_RESOURCE 
            # Condition: resource never present in history window,
            #            then present for NEW_RES_FUTURE_CONSEC consecutive snapshots.
            if float(np.sum(window_pres)) == 0:
                end = min(t + NEW_RES_FUTURE_CONSEC, n_snaps)
                consec_present = pres[t:end]
                if (len(consec_present) == NEW_RES_FUTURE_CONSEC
                        and float(np.sum(consec_present)) == NEW_RES_FUTURE_CONSEC
                        and (t == WINDOW or pres[t - 1] == 0.0)):
                    anomalies[dom].append((t, 'NEW_RESOURCE'))

            # ERROR_4XX
            baseline_4xx = float(np.median(e4[t - WINDOW: t]))
            if e4[t] > baseline_4xx:
                anomalies[dom].append((t, 'ERROR_4XX'))

            # ERROR_5XX
            baseline_5xx = float(np.median(e5[t - WINDOW: t]))
            if e5[t] > baseline_5xx:
                anomalies[dom].append((t, 'ERROR_5XX'))

    return anomalies, presence, err_4xx, err_5xx


print('Sliding window detection loaded.')


# VISUALISATION 

def plot_page(target, test_id, resources, resources_full_len, n_snaps, snapshots, presence, anomalies,
              adblocker_used, save_graphs, show_graphs, output_dir):
    BG, TC = '#0f172a', 'white'
    mat = np.array([presence[d] for d in resources])
    
    
    # keep the beginning and the last 12 characters of the URL.
    resources = [
        d if len(d) <= RESOURCE_URL_HEAD_SIZE 
        else d[:RESOURCE_URL_HEAD_SIZE ] + "..." + d[-12:]
        for d in resources] 
    
    max_label_len = max(len(d) for d in resources)
    label_space = max_label_len * 0.2
    
    
    fig_h = max(6, len(resources) * 0.25)
    fig_w = max(20, n_snaps * 0.012) + label_space
    fig, ax = plt.subplots(figsize=(min(fig_w, 60), min(fig_h, 80)))
    fig.patch.set_facecolor(BG)
    ax.set_facecolor(BG)
    
    cmap = LinearSegmentedColormap.from_list(
        'custom',
        ['#0f172a','white','green']
    )


    ax.imshow(
        mat,
        cmap=cmap,
        vmin=0, vmax=1,
        aspect='auto',
        origin='upper',
        interpolation='none',
    )

    ax.set_yticks(np.arange(len(resources)))
 
        
    ax.set_yticklabels(resources, fontsize=8, color=TC)

    LABEL_N = max(1, n_snaps // 20)
    x_ticks = [i for i in range(n_snaps) if i % LABEL_N == 0]
    x_labels = [snapshots[i]['timestamp'][:16] for i in x_ticks]
    ax.set_xticks(x_ticks)
    ax.set_xticklabels(x_labels, fontsize=7, color=TC, rotation=45, ha='right')

    dom_idx = {d: i for i, d in enumerate(resources)}
    plotted_types = set()
    for dom, events in anomalies.items():
        if dom not in dom_idx:
            continue
        y = dom_idx[dom]
        for snap_idx, atype in events:
            ax.scatter(snap_idx, y,
                       color=ANOM_COLORS[atype], s=25, zorder=5,
                       marker='o', alpha=0.9, edgecolors='none')
            plotted_types.add(atype)

    ax.axvline(WINDOW, color='#f59e0b', linewidth=1.2, linestyle='--', alpha=0.6, zorder=4)
    ax.text(WINDOW + 5, 0.3, f'Window ({WINDOW * 10 // 60} h)',
            color='#f59e0b', fontsize=7, va='bottom')

    handles = [mpatches.Patch(color=ANOM_COLORS[t], label=t)
               for t in ANOM_COLORS if t in plotted_types]
    if handles:
        ax.legend(handles=handles, loc='upper right', fontsize=8,
                  facecolor=BG, edgecolor='white', labelcolor=TC)

    total_anom = sum(len(v) for v in anomalies.values())
    counts = defaultdict(int)
    for events in anomalies.values():
        for _, atype in events:
            counts[atype] += 1
    counts_str = '  '.join(f'{t}: {counts[t]}' for t in ANOM_COLORS if counts[t])
    adblocker_label = 'Adblocker: ON' if adblocker_used else 'Adblocker: OFF'
    count_analysed_resources =  f'Analysed {len(resources)} resources' if len(resources) == resources_full_len else f'Analysed {len(resources)} / {resources_full_len} resources' 
    ax.set_title(
        f'{target}  |  TestId: {test_id}  |  Resource Analysis  |  {adblocker_label} | {count_analysed_resources}\n'
        f'Anomalies: {total_anom}   {counts_str}',
        color=TC, fontsize=12, pad=12, loc='left'
    )
    ax.set_ylabel('Resources', color=TC, fontsize=10)
    ax.tick_params(axis='x', colors=TC)
    ax.tick_params(axis='y', colors=TC)
    for spine in ax.spines.values():
        spine.set_color(TC)

    plt.tight_layout()
    out = None
    
    if save_graphs:
        host_slug = (target
                     .replace('https://', '').replace('http://', '')
                     .replace('.', '_').replace('/', '_'))
        ts = datetime.now().strftime('%Y%m%d_%H%M%S')
        out = output_dir / f'{ts}.{test_id}.{host_slug}.png'
        plt.savefig(out, dpi=130, bbox_inches='tight', facecolor=BG)

    if show_graphs:
        plt.show()

    plt.close()
    return out, total_anom, dict(counts)


print('Visualisation loaded.')

Aggregation function loaded.
Sliding window detection loaded.
Visualisation loaded.


In [4]:
# WIDGETS 

date_objects = [date.fromisoformat(d) for d in available_dates]
options = [(d.strftime(' %Y-%m-%d '), d) for d in date_objects]

date_range = widgets.SelectionRangeSlider(
    options=options,
    index=(0, len(options) - 1),
    description='Date range:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='900px')
)

host_labels = [e['target_host'] for e in data_config]

site_select = widgets.SelectMultiple(
    options=host_labels,
    value=host_labels,
    rows=min(len(host_labels), 12),
    description='Sites:',
    style={'description_width': '60px'},
    layout=widgets.Layout(width='500px')
)

btn_all_sites = widgets.Button(description='Select all', button_style='')
btn_none_sites = widgets.Button(description='Clear selection', button_style='')

ad_blocker = widgets.Checkbox(
    value=True,
    description='Use adblocker to filter ads',
    indent=False
)

save_graphs = widgets.Checkbox(
    value=True,
    description='Save graphs to ' + GRAPH_SAVE_FOLDER,
    indent=False
)

show_graphs = widgets.Checkbox(
    value=True,
    description='Show graphs in console',
    indent=False
)

run_btn = widgets.Button(
    description='▶ Run analysis',
    button_style='success',
    layout=widgets.Layout(width='200px', height='40px')
)

output = widgets.Output()


#BUTTON CALLBACKS

def on_all_sites(b):
    site_select.value = host_labels


def on_none_sites(b):
    site_select.value = []


def on_run(b):
    date_from, date_to = date_range.value
    selected_hosts = list(site_select.value)
    use_adblocker = ad_blocker.value
    selected_test_ids = {host_to_testid[h] for h in selected_hosts if h in host_to_testid}

    selected_dates = [
        d for d in available_dates
        if date.fromisoformat(d) >= date_from and date.fromisoformat(d) <= date_to
    ]
    selected_files = [
        p for p in json_files
        if (d := extract_date(p)) and d in selected_dates
    ]

    with output:
        output.clear_output(wait=True)

        print(f"Selected date range : {selected_dates[0]} – {selected_dates[-1]}")
        print(f"Selected sites      : {len(selected_test_ids)}")
        print(f"Using adblocker     : {use_adblocker}")

        # LOAD
        pages_raw = defaultdict(list)

        for filepath in selected_files:
            with open(filepath, encoding='utf-8') as f:
                print(f"Loading file to analyze ...{filepath}")
                for raw in f:
                    raw = raw.strip()
                    if not raw:
                        continue
                    try:
                        obj = json.loads(raw)
                    except json.JSONDecodeError:
                        continue

                    result = obj.get('Result')
                    if not result or result.get('status') != 'completed':
                        continue

                    pages = result.get('visited_pages', [])
                    if not pages:
                        continue

                    test_id = obj.get('Meta', {}).get('TestId', '')
                    if test_id not in selected_test_ids:
                        continue

                    host = testid_to_host.get(test_id, test_id)
                    timestamp = (obj.get('Meta', {}).get('Timestamp')
                                 or obj.get('Meta', {}).get('timestamp')
                                 or filepath.stem)

                    resources = []
                    for res in pages[0].get('waterfall_analysis', []):
                        method = res.get("method", "") or "METHOD_EMPTY"
                        url = res.get('url', '')
                        if not url.startswith('http'):
                            continue

                        resources.append({
                            'url': url,
                            'method': method,
                            'status': res.get('status_code', ''),
                            'source': res.get('sourceUrl', ''),
                            'type': res.get('type', 'other'),
                        })

                    pages_raw[host].append({
                        'timestamp': str(timestamp),
                        'test_id': test_id,
                        'resources': resources,
                    })

        print(f"\nLoaded sites: {len(pages_raw)}\n")
        for host, snaps in pages_raw.items():
            print(f"  {host:45s}  {len(snaps)} snapshots")

        # ANALYSE AND PLOT
        GRAPH_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
        ad_blocker_blocked_urls = None
        if use_adblocker:
            ad_blocker_processed_files = load_adblock_processed_files_cache()
            files_to_process = [x for x in selected_files if x.name not in ad_blocker_processed_files]
            if files_to_process:
                adblock_store_to_cache(files_to_process)
            ad_blocker_blocked_urls = load_adblock_cache()

        for target, snapshots in sorted(pages_raw.items()):
            test_id = host_to_testid.get(target, 'unknown')
            print(f'\n[{target}]  TestId: {test_id}  ({len(snapshots)} snapshots)')

            snap_data, resources, n_snaps = aggregate_page(snapshots, ad_blocker_blocked_urls)
            out_path = None

            if not resources or n_snaps <= WINDOW:
                print(f'[INFO] skipping (need > {WINDOW} snapshots, have {n_snaps})')
                continue

            resources_full_len = len(resources)
            resources = resources[:MAX_RESOURCES_TO_ANALYZE]
            if len(resources) < resources_full_len:
                print(
                    f'[INFO] Analyzing {len(resources)}/{resources_full_len} resources skipping {resources_full_len - len(resources)} resources')
            anomalies, presence, _, _ = detect_sliding_window(snap_data, resources, n_snaps)
            
            out_path, total_anom, counts = plot_page(
                target, test_id, resources, resources_full_len, n_snaps, snapshots,
                presence, anomalies, use_adblocker, save_graphs.value, show_graphs.value, GRAPH_RESULTS_DIR
            )

            counts_str = '  '.join(f'{t}: {counts.get(t, 0)}' for t in ANOM_COLORS if counts.get(t, 0))
            print(f'[INFO]  anomalies: {total_anom}   {counts_str}')
            if out_path:
                print(f'[INFO]  saved: {out_path.name}')

        print("Done.")
        if out_path:
            print(f'\nCharts saved to: {GRAPH_RESULTS_DIR.resolve()}')


# DISPLAY AND REGISTER CALLBACKS

btn_all_sites.on_click(on_all_sites)
btn_none_sites.on_click(on_none_sites)
run_btn.on_click(on_run)

display(
    widgets.Label('── Select Date Range ──'),
    date_range,
    widgets.Label('── Select Websites ──'),
    widgets.Label('Multiple values can be selected with Shift and/or Ctrl (or Cmd) + click.'),
    site_select,
    widgets.HBox([btn_all_sites, btn_none_sites]),
    ad_blocker,
    show_graphs,
    save_graphs,
    widgets.Label(''),
    run_btn,
    output
)

Label(value='── Select Date Range ──')

SelectionRangeSlider(description='Date range:', index=(0, 4), layout=Layout(width='900px'), options=((' 2026-0…

Label(value='── Select Websites ──')

Label(value='Multiple values can be selected with Shift and/or Ctrl (or Cmd) + click.')

SelectMultiple(description='Sites:', index=(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, …

...

Checkbox(value=True, description='Show graphs in console', indent=False)

Checkbox(value=True, description='Save graphs to ./results', indent=False)

Label(value='')

Button(button_style='success', description='▶ Run analysis', layout=Layout(height='40px', width='200px'), styl…

Output()